# Liam Pressure Pass-Through (Single Sample)

Goal: convert one wrangled composition into Liam's expected CSV format, then run Liam's pressure implementation as-is.

Use the Python 3.10 `enki` kernel for this notebook.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path('/Users/lopezama/PycharmProjects/sci-cluster')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sci_helpers import preflight_liam_runtime

preflight = preflight_liam_runtime(strict=False)
preflight

In [ ]:
WRANGLED_CSV = REPO_ROOT / 'sci-data/wrangled-outputs/wrangled_KCP-109-C_compositions.csv'
SAMPLE_ID = 'KCP-109-B'
RUN_ROOT = REPO_ROOT / 'outputs/liam-pressure-runs'
LIAM_INPUT_OUT = RUN_ROOT / 'manual-inputs' / f'liam_input_{WRANGLED_CSV.stem}__{SAMPLE_ID}.csv'

MELTS_PARAMS = {
    'Model': 'rhyolite-MELTS_v1.0.x',
    'Calculation': 'QF_P_Calc',
    'T1': 1100,
    'T2': 700,
    'ΔT': 1,
    'T unit': 'C',
    'P1': 400,
    'P2': 50,
    'ΔP': 25,
    'P unit': 'MPa',
    'fO2 offset': 0,
    'fO2 buffer': 'NNO',
    'fO2 constraint': 'TRUE',
    'ΔH': 0.5,
    'ΔV': 0,
    'ΔS': 0,
}

FIXED_OXIDE_OVERRIDES = {
    'H2O': 13.0,
    'Fe2O3': 0.18,
    'Cr2O3': 0.0,
    'NiO': 0.0,
    'CoO': 0.0,
    'CO2': 0.0,
    'SO3': 0.0,
    'Cl2O-1': 0.0,
    'F2O-1': 0.0,
}

PRESSURE_WORKERS = 1
VERBOSE = True

In [ ]:
import pandas as pd
from sci_helpers import build_and_write_liam_input_for_sample

liam_csv = build_and_write_liam_input_for_sample(
    wrangled_csv=WRANGLED_CSV,
    sample_id=SAMPLE_ID,
    melts_params=MELTS_PARAMS,
    output_csv=LIAM_INPUT_OUT,
    fixed_oxide_overrides=FIXED_OXIDE_OVERRIDES,
)

liam_df = pd.read_csv(liam_csv, index_col=0)
print('Liam input CSV:', liam_csv)
liam_df.head(40)

In [ ]:
from sci_helpers import PreparedInputRunConfig, run_single_liam_pressure_from_prepared_csv

cfg = PreparedInputRunConfig(
    prepared_liam_csv=liam_csv,
    max_composition_workers=1,
    max_pressure_workers=PRESSURE_WORKERS,
    vendor_code_dir=REPO_ROOT / 'vendor/LeiTesting',
    run_root_dir=RUN_ROOT,
    verbose=VERBOSE,
)

summary = run_single_liam_pressure_from_prepared_csv(cfg)
summary

In [ ]:
import json
import shutil

summary_view = {k: v for k, v in summary.items() if k not in {'results', 'preflight'}}
print(json.dumps(summary_view, indent=2))

results_df = pd.DataFrame(summary['results'])
display(results_df)

download_copy = Path('/Users/lopezama/Downloads') / Path(liam_csv).name
shutil.copy2(liam_csv, download_copy)
print('Copied Liam input CSV to:', download_copy)